In [8]:
import json
import os
import numpy as np
import pandas as pd
import torch

from chronos import BaseChronosPipeline
from sklearn.metrics import root_mean_squared_error

# ============================================================
# CONFIG
# ============================================================
DAYS_JSON = r"C:\Users\CR58XM\Documents\GitHub\AAU_learning_to_predict_together_or_alone\dataset_days.json"
DATA_DIR  = r"C:\Users\CR58XM\Documents\GitHub\AAU_learning_to_predict_together_or_alone\DataCleaning\clean"
OUT_DIR   = r"C:\Users\CR58XM\Documents\GitHub\AAU_learning_to_predict_together_or_alone\Outputs"

countries = ["Germany", "Ireland", "Portugal"]
days = ["day1", "day2", "day3", "day4", "day5"]

features = [
    "temperature_2m",
    "relative_humidity_2m",
    "wind_speed_10m",
    "precipitation",
    "direct_radiation",
]

PRED_LEN = 96
CONTEXT_TAIL = 10000
QUANTILES = [0.5]  # median

# Optional: reduce CPU threads if you want your PC responsive
# torch.set_num_threads(8)

# ============================================================
# LOAD SPLITS
# ============================================================
with open(DAYS_JSON, "r") as f:
    dataset_days = json.load(f)

# ============================================================
# LOAD CHRONOS-T5-LARGE ON CPU
# ============================================================
pipeline = BaseChronosPipeline.from_pretrained(
    "amazon/chronos-t5-large",
    device_map="cpu",
)
print("torch:", torch.__version__)
print("pipeline model device:", pipeline.model.device)

rmse_results = []

for country in countries:
    print("\nProcessing country:", country)

    data_path = os.path.join(DATA_DIR, f"dataset_{country.capitalize()}.csv")
    df = pd.read_csv(data_path, index_col="timestamp", parse_dates=True).sort_index()

    households = [c for c in df.columns if c not in features]

    for day in days:
        print("   Day:", day)
        cutoff = pd.to_datetime(dataset_days[country][day])

        # Prefer real timestamps from data for alignment
        future_index = df.loc[df.index >= cutoff].index[:PRED_LEN]
        if len(future_index) < PRED_LEN:
            inferred = pd.infer_freq(df.index)
            if inferred is None:
                deltas = df.index.to_series().diff().dropna()
                step = deltas.mode().iloc[0]
                future_index = pd.date_range(start=cutoff, periods=PRED_LEN, freq=step)
            else:
                future_index = pd.date_range(start=cutoff, periods=PRED_LEN, freq=inferred)

        predictions_df_all_households = pd.DataFrame(index=future_index)
        predictions_df_all_households.index.name = "timestamp"

        rmse_households = []

        for household in households:
            # history slice (univariate)
            s_train = df.loc[df.index < cutoff, household].astype(float).dropna()

            if len(s_train) < 10:
                predictions_df_all_households[household] = np.nan
                continue

            s_train = s_train.tail(CONTEXT_TAIL)

            # IMPORTANT: keep inputs on CPU
            inputs = torch.tensor(s_train.values, dtype=torch.float32)

            with torch.no_grad():
                quantiles, mean = pipeline.predict_quantiles(
                    inputs=inputs,
                    prediction_length=PRED_LEN,
                    quantile_levels=QUANTILES,
                )

            # Handle possible output shapes
            if quantiles.ndim == 2:         # [pred_len, num_q]
                y_pred = quantiles[:, 0].detach().numpy()
            else:                            # [1, pred_len, num_q]
                y_pred = quantiles[0, :, 0].detach().numpy()

            predictions_df_all_households[household] = y_pred

            # RMSE (only where truth exists)
            y_true = df.loc[predictions_df_all_households.index, household].astype(float)
            y_pred_s = predictions_df_all_households[household]

            mask = y_true.notna() & y_pred_s.notna()
            if mask.sum() > 0:
                rmse = root_mean_squared_error(y_true[mask], y_pred_s[mask])
                rmse_households.append(rmse)

        avg_rmse = float(np.mean(rmse_households)) if rmse_households else np.nan
        rmse_results.append({"country": country, "day": day, "rmse": avg_rmse})

        out_path = os.path.join(
            OUT_DIR,
            f"Chronos_t5large_Univar_pred_{day}_{country.capitalize()}.csv"
        )
        os.makedirs(os.path.dirname(out_path), exist_ok=True)
        predictions_df_all_households.to_csv(out_path, index=True)
        print("      Saved:", out_path)

# ============================================================
# SUMMARY
# ============================================================
rmse_df = pd.DataFrame(rmse_results)
print("\nPer-day RMSE:")
print(rmse_df)

print("\nCross-validated RMSE per country (mean over days):")
print(rmse_df.groupby("country")["rmse"].mean())

torch: 2.5.1+cu121
pipeline model device: cpu

Processing country: Germany


We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 


   Day: day1


We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of l

      Saved: C:\Users\CR58XM\Documents\GitHub\AAU_learning_to_predict_together_or_alone\Outputs\Chronos_t5large_Univar_pred_day1_Germany.csv
   Day: day2


...

      Saved: C:\Users\CR58XM\Documents\GitHub\AAU_learning_to_predict_together_or_alone\Outputs\Chronos_t5large_Univar_pred_day3_Portugal.csv
   Day: day4


We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of l

      Saved: C:\Users\CR58XM\Documents\GitHub\AAU_learning_to_predict_together_or_alone\Outputs\Chronos_t5large_Univar_pred_day4_Portugal.csv
   Day: day5


We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of l

      Saved: C:\Users\CR58XM\Documents\GitHub\AAU_learning_to_predict_together_or_alone\Outputs\Chronos_t5large_Univar_pred_day5_Portugal.csv

Per-day RMSE:
     country   day         rmse
0    Germany  day1  1411.162267
1    Germany  day2   252.518541
2    Germany  day3   722.527071
3    Germany  day4  1401.690689
4    Germany  day5   274.423207
5    Ireland  day1   926.735302
6    Ireland  day2   189.466272
7    Ireland  day3   654.478408
8    Ireland  day4   883.816309
9    Ireland  day5   479.528680
10  Portugal  day1   457.119264
11  Portugal  day2   297.610502
12  Portugal  day3   378.746027
13  Portugal  day4   422.390322
14  Portugal  day5   201.101932

Cross-validated RMSE per country (mean over days):
country
Germany     812.464355
Ireland     626.804994
Portugal    351.393609
Name: rmse, dtype: float64
